# Painel de Análise de Vendas com Visualização de Dados

Atividade completa das páginas 67 e 68.  
As etapas 1 a 20 foram implementadas com pandas, numpy, matplotlib e seaborn.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

os.makedirs("graficos", exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## Etapa 1 — Preparação e Importação de Dados

In [ ]:
# Etapas 2 e 3
df = pd.read_csv("vendas.csv")
df["data"] = pd.to_datetime(df["data"])
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
display(df.head())


In [ ]:
# Etapa 4 — Integridade
df.info()
print("\nValores ausentes:")
display(df.isnull().sum())
print("\nResumo estatístico:")
display(df.describe(include="all"))


In [ ]:
# Etapa 5 — Substituição dos nulos pela média da respectiva coluna
df["quantidade"] = df["quantidade"].fillna(df["quantidade"].mean())
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
display(df.isnull().sum())


## Etapa 2 — Análise Exploratória e Estatísticas

In [ ]:
# Etapa 6
faturamento_loja = df.groupby("loja", as_index=False)["faturamento"].sum().sort_values("faturamento", ascending=False)
display(faturamento_loja)


In [ ]:
# Etapa 7
ticket_medio = df.groupby("loja").agg(faturamento=("faturamento","sum"), vendas=("produto","count"))
ticket_medio["ticket_medio"] = ticket_medio["faturamento"] / ticket_medio["vendas"]
ticket_medio = ticket_medio.reset_index().sort_values("ticket_medio", ascending=False)
display(ticket_medio)


In [ ]:
# Etapa 8
top5_produtos = df.groupby("produto", as_index=False)["quantidade"].sum().sort_values("quantidade", ascending=False).head(5)
display(top5_produtos)


In [ ]:
# Etapas 9 e 10
df["mes"] = df["data"].dt.month
df["mes_ano"] = df["data"].dt.to_period("M").astype(str)
faturamento_mes = df.groupby("mes_ano", as_index=False)["faturamento"].sum()
faturamento_mes["data_mes"] = pd.to_datetime(faturamento_mes["mes_ano"] + "-01")
faturamento_mes = faturamento_mes.sort_values("data_mes")
maior_mes = faturamento_mes.loc[faturamento_mes["faturamento"].idxmax()]
menor_mes = faturamento_mes.loc[faturamento_mes["faturamento"].idxmin()]
print("Maior faturamento:", maior_mes["data_mes"].strftime("%m/%Y"))
print("Menor faturamento:", menor_mes["data_mes"].strftime("%m/%Y"))
display(faturamento_mes)


## Etapa 3 — Visualizações com matplotlib e seaborn

In [ ]:
# Etapa 17
sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))


In [ ]:
# Etapa 11 — Barras
plt.figure(figsize=(10,6))
ax=sns.barplot(data=faturamento_loja,x="loja",y="faturamento",hue="loja",legend=False)
plt.title("Faturamento Total por Loja"); plt.xlabel("Loja"); plt.ylabel("Faturamento (R$)")
plt.xticks(rotation=20)
for c in ax.containers: ax.bar_label(c,fmt="R$ %.0f")
plt.tight_layout(); plt.savefig("graficos/01_faturamento_por_loja.png",dpi=150); plt.show()


In [ ]:
# Etapa 12 — Linhas
plt.figure(figsize=(11,6))
ax=sns.lineplot(data=faturamento_mes,x="data_mes",y="faturamento",marker="o",linewidth=2)
plt.title("Evolução Mensal do Faturamento"); plt.xlabel("Mês"); plt.ylabel("Faturamento (R$)")
for _,r in faturamento_mes.iterrows():
    ax.annotate(f"R$ {r['faturamento']:,.0f}".replace(",", "."),(r["data_mes"],r["faturamento"]),textcoords="offset points",xytext=(0,8),ha="center",fontsize=7)
plt.xticks(faturamento_mes["data_mes"],[d.strftime("%b/%Y") for d in faturamento_mes["data_mes"]],rotation=45)
plt.tight_layout(); plt.savefig("graficos/02_evolucao_mensal.png",dpi=150); plt.show()


In [ ]:
# Etapa 13 — Pizza
df["regiao"]=df["loja"].map({"Loja Centro":"Sudeste","Loja Norte":"Norte","Loja Sul":"Sul","Loja Leste":"Nordeste"})
faturamento_regiao=df.groupby("regiao",as_index=False)["faturamento"].sum().sort_values("faturamento",ascending=False)
plt.figure(figsize=(8,8))
plt.pie(faturamento_regiao["faturamento"],labels=faturamento_regiao["regiao"],autopct="%1.1f%%",startangle=90)
plt.title("Distribuição do Faturamento por Região"); plt.tight_layout()
plt.savefig("graficos/03_faturamento_por_regiao.png",dpi=150); plt.show()


In [ ]:
# Etapa 14 — Dispersão
plt.figure(figsize=(10,6))
sns.scatterplot(data=df,x="quantidade",y="faturamento",hue="categoria",style="categoria",s=70,alpha=.75)
plt.title("Relação entre Quantidade Vendida e Faturamento"); plt.xlabel("Quantidade de Itens Vendidos"); plt.ylabel("Faturamento (R$)")
plt.legend(title="Categoria",bbox_to_anchor=(1.02,1),loc="upper left")
plt.tight_layout(); plt.savefig("graficos/04_quantidade_vs_faturamento.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# Etapa 15 — Boxplot
plt.figure(figsize=(10,6))
sns.boxplot(data=df,x="categoria",y="preco_unitario",hue="categoria",legend=False)
plt.title("Distribuição dos Preços Unitários por Categoria"); plt.xlabel("Categoria"); plt.ylabel("Preço Unitário (R$)")
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig("graficos/05_precos_por_categoria.png",dpi=150); plt.show()


In [ ]:
# Etapa 16 — Heatmap
corr=df[["quantidade","preco_unitario","faturamento","mes"]].corr()
plt.figure(figsize=(9,7))
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm",linewidths=.5)
plt.title("Matriz de Correlação das Variáveis Numéricas"); plt.tight_layout()
plt.savefig("graficos/06_matriz_correlacao.png",dpi=150); plt.show()


## Etapa 4 — Estilização e Apresentação

In [ ]:
# Etapas 18 e 19 — Layout comparativo
fig,axes=plt.subplots(1,2,figsize=(15,6))
sns.barplot(data=faturamento_loja,x="loja",y="faturamento",hue="loja",legend=False,ax=axes[0])
axes[0].set_title("Faturamento por Loja"); axes[0].set_xlabel("Loja"); axes[0].set_ylabel("Faturamento (R$)"); axes[0].tick_params(axis="x",rotation=20)
sns.barplot(data=faturamento_regiao,x="regiao",y="faturamento",hue="regiao",legend=False,ax=axes[1])
axes[1].set_title("Faturamento por Região"); axes[1].set_xlabel("Região"); axes[1].set_ylabel("Faturamento (R$)"); axes[1].tick_params(axis="x",rotation=20)
fig.suptitle("Comparativo de Faturamento: Lojas e Regiões",fontsize=15)
plt.tight_layout(); plt.savefig("graficos/07_comparativo_lojas_regioes.png",dpi=150); plt.show()


## Etapa 20 — Exportação e relatório final

In [ ]:
resumo=pd.DataFrame([
    ["Maior faturamento por loja",faturamento_loja.iloc[0]["loja"],faturamento_loja.iloc[0]["faturamento"]],
    ["Menor faturamento por loja",faturamento_loja.iloc[-1]["loja"],faturamento_loja.iloc[-1]["faturamento"]],
    ["Maior ticket médio",ticket_medio.iloc[0]["loja"],ticket_medio.iloc[0]["ticket_medio"]],
    ["Menor ticket médio",ticket_medio.iloc[-1]["loja"],ticket_medio.iloc[-1]["ticket_medio"]],
    ["Mês de maior faturamento",maior_mes["data_mes"].strftime("%m/%Y"),maior_mes["faturamento"]],
    ["Mês de menor faturamento",menor_mes["data_mes"].strftime("%m/%Y"),menor_mes["faturamento"]],
    ["Região com maior faturamento",faturamento_regiao.iloc[0]["regiao"],faturamento_regiao.iloc[0]["faturamento"]],
    ["Produto mais vendido",top5_produtos.iloc[0]["produto"],top5_produtos.iloc[0]["quantidade"]]
],columns=["indicador","categoria","valor"])
resumo.to_csv("resumo_analitico.csv",index=False,encoding="utf-8-sig")
faturamento_loja.to_csv("faturamento_por_loja.csv",index=False,encoding="utf-8-sig")
ticket_medio.to_csv("ticket_medio_por_loja.csv",index=False,encoding="utf-8-sig")
top5_produtos.to_csv("top5_produtos.csv",index=False,encoding="utf-8-sig")
faturamento_regiao.to_csv("faturamento_por_regiao.csv",index=False,encoding="utf-8-sig")
display(resumo)
